# 04 forward + logits + KV cache

目标：不调用 `generate()`，只跑一次 forward，观察 logits 和 KV cache。


## 运行环境准备

这个 notebook 默认使用 `Qwen/Qwen2.5-0.5B-Instruct`，适合在魔搭 Notebook 里快速学习。

如果你想用更大的模型，可以把 `MODEL_ID` 改成 `Qwen/Qwen2.5-7B-Instruct`，然后重启内核重新运行。


In [2]:
from pathlib import Path

requirements_path = Path("requirements.txt")
if not requirements_path.exists():
    requirements_path = Path("../requirements.txt")

%pip install -r {requirements_path}


ERROR: Could not open requirements file: [Errno 2] 没有那个文件或目录: '../requirements.txt'

[notice] A new release of pip is available: 23.3.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from pathlib import Path

MODEL_ID = os.getenv("MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
MODEL_SOURCE = os.getenv("MODEL_SOURCE", "modelscope").lower()


def resolve_model_path(model_id):
    if Path(model_id).exists():
        return model_id
    if MODEL_SOURCE != "modelscope":
        return model_id

    from modelscope import snapshot_download
    return snapshot_download(model_id)


MODEL_PATH = resolve_model_path(MODEL_ID)
print("MODEL_ID =", MODEL_ID)
print("MODEL_PATH =", MODEL_PATH)


/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-05-20 23:15:18,267 - modelscope - INFO - Target directory already exists, skipping creation.


MODEL_ID = Qwen/Qwen2.5-0.5B-Instruct
MODEL_PATH = /mnt/workspace/.cache/modelscope/models/Qwen/Qwen2___5-0___5B-Instruct


## 1. 加载模型并准备输入


In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True,
)

messages = [
    {"role": "user", "content": "KV cache 是什么？"},
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)


Loading weights: 100%|██████████| 290/290 [00:09<00:00, 30.14it/s]


## 2. 单次 forward

`logits[:, -1, :]` 表示最后一个输入位置对整个词表的预测分数。


In [5]:
with torch.no_grad():
    outputs = model(**inputs, use_cache=True)

logits = outputs.logits[:, -1, :]
next_token_id = logits.argmax(dim=-1)
next_token = tokenizer.decode(next_token_id)

print("input token count:", inputs["input_ids"].shape[-1])
print("vocab size:", logits.shape[-1])
print("greedy next token id:", next_token_id.item())
print("greedy next token:", repr(next_token))


input token count: 34
vocab size: 151936
greedy next token id: 82707
greedy next token: 'KV'


## 3. 查看 KV cache

KV cache 会保存每层 attention 的 key/value，长上下文和高并发时会占用大量显存。


In [7]:
cache = outputs.past_key_values
print(cache)
print("KV cache type:", type(cache).__name__)

try:
    print("KV cache layers:", len(cache))
except TypeError:
    print("KV cache layers: unknown")

if hasattr(cache, "get_seq_length"):
    print("KV cache sequence length:", cache.get_seq_length())


DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer])
KV cache type: DynamicCache
KV cache layers: 24
KV cache sequence length: 34
